# Build Resized Dataset

Reads the metadata CSV produced by `1.dataset_stats.ipynb`, resizes every image to a uniform square resolution (default: 336x336 px), normalizes class folder names, and saves the resulting dataset to a new directory tree organised as `<output_root>/<team>/<class>/`.

Processing strategy:
- Square images: direct resize with bicubic interpolation.
- Non-square images: scale the shorter edge to the target size, then centre-crop.
- Images smaller than the target: upscaled before cropping.
- Output filenames: perceptual hash (from metadata CSV) with the chosen extension.

**Input:** `image_dataset_summary_with_hash.csv`  
**Outputs:** resized image tree, `resizing_report.csv`, `dataset_summary_clean.csv`, `dataset_statistics.csv`


In [ ]:
%pip install pandas Pillow pillow-heif

import os
import csv
import pandas as pd
import numpy as np
from PIL import Image, ImageFile
from pillow_heif import register_heif_opener
from datetime import datetime
import warnings

register_heif_opener()
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", "(Possibly )?corrupt EXIF data", UserWarning)

print("Libraries imported and configured.")


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.
✅ Libraries imported and configured successfully.


## Configuration

Set paths, target resolution, output format, and class name mapping here before running the rest of the notebook.


In [2]:
# --- Main Configuration ---
# Path to the CSV file containing the image paths and hashes.
METADATA_CSV_PATH = 'image_dataset_summary_with_hash.csv'

# --- Image Processing Parameters ---
# Target resolution (width and height) for all output images.
TARGET_SIZE = 336

# --- Output directory structure: Dataset/<Team_name>/<Class_name> ---
OUTPUT_ROOT_DIR = f"Dataset_w{TARGET_SIZE}_h{TARGET_SIZE}_{datetime.now().strftime('%Y-%m-%d')}/"

# Output format: "JPEG" for high-quality lossy, or "PNG" for lossless.
OUTPUT_FORMAT = "JPEG"
JPEG_QUALITY = 95 # Quality setting for JPEG saving.

# --- Class Normalization Mapping ---
# Maps various class folder names to a single, normalized name.
# Keys should be lowercase as the script will clean names before mapping.
CLASS_MAPPING = {
    "tipu": "tipu",
    "chenes": "chene",
    "chene": "chene",
    "frenes": "frene",
    "frene": "frene",
    "caroubier": "caroubier",
    "faux_poivrier": "faux_poivrier",
    "pistachier": "pistachier"
}

# --- Initialization ---
OUTPUT_EXTENSION = ".jpg" if OUTPUT_FORMAT == "JPEG" else ".png"
resizing_report_data = []
summary_data = []
corrupted_files_data = []
original_resolutions = []

print(f"✅ Configuration set. Output will be saved to '{OUTPUT_ROOT_DIR}'")

✅ Configuration set. Output will be saved to 'Dataset_w336_h336_2025-08-15/'


## Processing Pipeline

Reads the metadata CSV, iterates over each image, applies resizing/cropping, and saves the result. Progress is printed to the console.


In [3]:
def get_file_size_mb(file_path):
    """Get file size in MB"""
    try:
        return round(os.path.getsize(file_path) / (1024 * 1024), 2)
    except:
        return 0

def extract_team_and_path_info(input_path):
    """Extract team name and handle subfolder structure"""
    path_parts = input_path.split(os.sep)
    
    # Find 'Teams_space' index to locate team info
    teams_space_idx = -1
    for i, part in enumerate(path_parts):
        if 'Teams_space' in part:
            teams_space_idx = i
            break
    
    if teams_space_idx == -1:
        # Fallback: assume structure is at least 3 levels deep
        team_name = path_parts[-3] if len(path_parts) >= 3 else "Unknown"
        class_name = path_parts[-2]
        extra_subfolder = ""
    else:
        # Extract team (next after Teams_space)
        team_name = path_parts[teams_space_idx + 1] if teams_space_idx + 1 < len(path_parts) else "Unknown"
        
        # Check if there's an extra subfolder between team and image file
        remaining_parts = path_parts[teams_space_idx + 2:-1]  # Everything between team and filename
        
        if len(remaining_parts) == 1:
            # Standard: Dataset/<Team>/<Class>/image.jpg
            class_name = remaining_parts[0]
            extra_subfolder = ""
        elif len(remaining_parts) == 2:
            # Extra subfolder: Dataset/<Team>/<Class>/<Subfolder>/image.jpg
            class_name = remaining_parts[0]
            extra_subfolder = remaining_parts[1]
        else:
            # Fallback for complex structures
            class_name = remaining_parts[0] if remaining_parts else "Unknown"
            extra_subfolder = "/".join(remaining_parts[1:]) if len(remaining_parts) > 1 else ""
    
    return team_name.strip(), class_name.strip(), extra_subfolder.strip()

# Check if the metadata file exists before starting
if not os.path.isfile(METADATA_CSV_PATH):
    print(f"❌ FATAL ERROR: Metadata file not found at '{METADATA_CSV_PATH}'. Please check the path in Cell 2.")
else:
    print("🚀 Starting image processing pipeline...")
    
    # Load the metadata from the CSV file
    df_meta = pd.read_csv(METADATA_CSV_PATH)
    print(f"   - Loaded {len(df_meta)} records from '{METADATA_CSV_PATH}'.")

    # Create the main output directory
    if not os.path.isdir(OUTPUT_ROOT_DIR):
        os.makedirs(OUTPUT_ROOT_DIR)
        print(f"   - Created output directory: '{OUTPUT_ROOT_DIR}'")

    # Iterate over each row in the metadata file
    total_images = len(df_meta)
    for index, row in df_meta.iterrows():
        # Simple progress indicator
        print(f"\r-> Processing image {index + 1}/{total_images}...", end="")
        
        is_corrupted = False
        error_message = ""
        
        try:
            input_path = os.path.normpath(row['image_path'])
            image_hash = row['image_hash']
            
            # Extract team and path information
            team_name, original_class, extra_subfolder = extract_team_and_path_info(input_path)
            
            # Normalize class name
            normalized_class = original_class.strip().lower()
            normalized_class = CLASS_MAPPING.get(normalized_class, normalized_class)
            
            # Construct output path: Dataset/<Team_name>/<Class_name>
            output_dir = os.path.join(OUTPUT_ROOT_DIR, team_name, normalized_class)
            output_path = os.path.join(output_dir, f"{image_hash}{OUTPUT_EXTENSION}")

            if not os.path.isfile(input_path):
                raise FileNotFoundError(f"Source file not found at {input_path}")

            # Get original file info
            original_size_mb = get_file_size_mb(input_path)
            original_extension = os.path.splitext(input_path)[1]
            original_filename = os.path.basename(input_path)

            with Image.open(input_path) as img:
                original_format = img.format
                original_res = img.size
                img_rgb = img.convert('RGB')
                w, h = img_rgb.size
                original_resolutions.append((w, h))

                resizing_method = "unknown"
                if w < TARGET_SIZE or h < TARGET_SIZE: resizing_method = "upscale_and_crop"
                
                if w == h:
                    if resizing_method != "upscale_and_crop": resizing_method = "square_resize"
                    resized_img = img_rgb.resize((TARGET_SIZE, TARGET_SIZE), Image.Resampling.BICUBIC)
                else:
                    if resizing_method != "upscale_and_crop": resizing_method = "rectangular_crop"
                    
                    if w < h: # Portrait
                        temp_img = img_rgb.resize((TARGET_SIZE, int(h * TARGET_SIZE / w)), Image.Resampling.BICUBIC)
                        left, top = 0, (temp_img.height - TARGET_SIZE) / 2
                    else: # Landscape
                        temp_img = img_rgb.resize((int(w * TARGET_SIZE / h), TARGET_SIZE), Image.Resampling.BICUBIC)
                        left, top = (temp_img.width - TARGET_SIZE) / 2, 0
                        
                    resized_img = temp_img.crop((left, top, left + TARGET_SIZE, top + TARGET_SIZE))

            # Create output directory and save
            os.makedirs(output_dir, exist_ok=True)
            resized_img.save(output_path, format=OUTPUT_FORMAT, quality=JPEG_QUALITY if OUTPUT_FORMAT == 'JPEG' else None)
            
            # Get new file size
            new_size_mb = get_file_size_mb(output_path)

            # Add to resizing report
            resizing_report_data.append({
                "original_path": input_path, "original_resolution": f"{original_res[0]}x{original_res[1]}",
                "new_path": output_path, "new_resolution": f"{TARGET_SIZE}x{TARGET_SIZE}",
                "resizing_method": resizing_method, "format_change": f"{original_format or 'N/A'} -> {OUTPUT_FORMAT}"
            })

        except Exception as e:
            is_corrupted = True
            error_message = str(e)
            image_name = os.path.basename(row.get('image_path', 'N/A'))
            print(f"\n    [!] Warning: Could not process file '{image_name}'. Marked as corrupted. Error: {e}")
            corrupted_files_data.append({"original_path": row.get('image_path', 'N/A'), "error_message": str(e)})
            
            # Set default values for failed processing
            input_path = row.get('image_path', 'N/A')
            team_name, original_class, extra_subfolder = extract_team_and_path_info(input_path)
            normalized_class = original_class.strip().lower()
            normalized_class = CLASS_MAPPING.get(normalized_class, normalized_class)
            original_size_mb = 0
            original_extension = os.path.splitext(input_path)[1] if input_path != 'N/A' else ''
            original_filename = os.path.basename(input_path) if input_path != 'N/A' else 'N/A'
            output_path = 'N/A'
            new_size_mb = 0
            w, h = 0, 0

        # Add to summary data (for both successful and failed processing)
        summary_data.append({
            "old_image_path": input_path,
            "old_image_name": original_filename,
            "team": team_name,
            "class": normalized_class,
            "old_extension": original_extension,
            "image_hash": row.get('image_hash', ''),
            "old_size_mb": original_size_mb,
            "old_width": w,
            "old_height": h,
            "device": row.get('device', ''),
            "metadata": row.get('metadata', ''),
            "corrupted": is_corrupted,
            "new_image_path": output_path,
            "New_width": TARGET_SIZE if not is_corrupted else 0,
            "New_height": TARGET_SIZE if not is_corrupted else 0,
            "new_image_size": new_size_mb,
            "Extra_subfolder": extra_subfolder
        })
            
    # Convert report list to a DataFrame
    df_report = pd.DataFrame(resizing_report_data)
    df_summary = pd.DataFrame(summary_data)
    print("\n\n✅ Processing complete!")

🚀 Starting image processing pipeline...
   - Loaded 47369 records from 'image_dataset_summary_with_hash.csv'.
   - Created output directory: 'Dataset_w336_h336_2025-08-15/'
-> Processing image 2848/47369...
    [!] Warning: Could not process file '20240625_094625.jpg'. Marked as corrupted. Error: cannot identify image file 'Agrichallenge_dataset/AgrI_Dataset/Teams_space/AiGro/faux_poivrier /20240625_094625.jpg'
-> Processing image 22004/47369...
    [!] Warning: Could not process file 'C500.jpg'. Marked as corrupted. Error: cannot identify image file 'Agrichallenge_dataset/AgrI_Dataset/Teams_space/RUSTICUS/caroubier/C500.jpg'
-> Processing image 47369/47369...

✅ Processing complete!


## Reports and Summary Statistics

Generates CSV reports and prints a final statistical overview of the processed dataset.


In [4]:
if not df_summary.empty:
    print("\n📊 Generating Reports & Statistics 📊")
    print("-----------------------------------")

    # --- Corrupted Files Report ---
    if corrupted_files_data:
        corrupted_df = pd.DataFrame(corrupted_files_data)
        corrupted_report_path = os.path.join(OUTPUT_ROOT_DIR, "corrupted_files_report.csv")
        corrupted_df.to_csv(corrupted_report_path, index=False)
        print(f"⚠️ Found {len(corrupted_df)} corrupted/skipped files. Report saved to '{corrupted_report_path}'")
    else:
        print("👍 No corrupted or skipped files were found.")

    # --- Main Resizing Report ---
    if not df_report.empty:
        main_report_path = os.path.join(OUTPUT_ROOT_DIR, "resizing_report.csv")
        df_report.to_csv(main_report_path, index=False)
        print(f"✅ Successfully processed {len(df_report)} images. Main report saved to '{main_report_path}'")

    # --- Complete Summary File (including corrupted entries) ---
    summary_report_path = os.path.join(OUTPUT_ROOT_DIR, "dataset_summary_resized.csv")
    df_summary.to_csv(summary_report_path, index=False)
    print(f"📋 Summary dataset file (including corrupted entries) saved to '{summary_report_path}'")
    
    # --- Clean Summary File (only successfully processed images) ---
    df_clean_summary = df_summary[df_summary['corrupted'] == False].copy()
    clean_summary_path = os.path.join(OUTPUT_ROOT_DIR, "dataset_summary_clean.csv")
    df_clean_summary.to_csv(clean_summary_path, index=False)
    print(f"📋 Clean summary file (only successfully processed images) saved to '{clean_summary_path}'")

    # --- Display Final Statistics ---
    if original_resolutions:
        resolutions_np = np.array(original_resolutions)
        print("\n--- Final Dataset Statistics ---")
        print(f"Total Images Processed: {len(df_report)}")
        print(f"Total Images Skipped: {len(corrupted_files_data)}")
        mean_res_orig = f"{int(np.mean(resolutions_np[:, 0]))}x{int(np.mean(resolutions_np[:, 1]))}"
        print(f"Original Mean Resolution: {mean_res_orig}")
        print(f"New Uniform Resolution: {TARGET_SIZE}x{TARGET_SIZE}")
    
    # --- Additional Summary Statistics ---
    print(f"\n--- Summary File Statistics ---")
    print(f"Total Records in Summary: {len(df_summary)}")
    print(f"Successfully Processed: {len(df_clean_summary)}")
    print(f"Teams Found: {df_summary['team'].nunique()}")
    print(f"Classes Found: {df_summary['class'].nunique()}")
    if 'Extra_subfolder' in df_summary.columns:
        extra_subfolders = df_summary[df_summary['Extra_subfolder'] != '']['Extra_subfolder'].nunique()
        print(f"Images with Extra Subfolders: {len(df_summary[df_summary['Extra_subfolder'] != ''])}")

    print("\n🎉 Pipeline Complete.")
else:
    print("\n❌ No images were processed. Check logs in the previous cell for errors.")


📊 Generating Reports & Statistics 📊
-----------------------------------
⚠️ Found 2 corrupted/skipped files. Report saved to 'Dataset_w336_h336_2025-08-15/corrupted_files_report.csv'
✅ Successfully processed 47367 images. Main report saved to 'Dataset_w336_h336_2025-08-15/resizing_report.csv'
📋 Summary dataset file (including corrupted entries) saved to 'Dataset_w336_h336_2025-08-15/dataset_summary_resized.csv'
📋 Clean summary file (only successfully processed images) saved to 'Dataset_w336_h336_2025-08-15/dataset_summary_clean.csv'

--- Final Dataset Statistics ---
Total Images Processed: 47367
Total Images Skipped: 2
Original Mean Resolution: 2709x2738
New Uniform Resolution: 336x336

--- Summary File Statistics ---
Total Records in Summary: 47369
Successfully Processed: 47367
Teams Found: 11
Classes Found: 6
Images with Extra Subfolders: 6321

🎉 Pipeline Complete.


In [5]:
# --- Optional: Dataset Analysis and Validation ---
print("🔍 Dataset Analysis and Validation")
print("=" * 50)

if not df_summary.empty:
    # --- Team and Class Distribution ---
    print("\n📊 Team Distribution:")
    team_counts = df_clean_summary['team'].value_counts()
    for team, count in team_counts.items():
        print(f"  • {team}: {count} images")
    
    print(f"\n📊 Class Distribution:")
    class_counts = df_clean_summary['class'].value_counts()
    for cls, count in class_counts.items():
        print(f"  • {cls}: {count} images")
    
    # --- Team-Class Cross Analysis ---
    print(f"\n📊 Team-Class Distribution:")
    team_class_pivot = df_clean_summary.pivot_table(
        index='team', 
        columns='class', 
        values='image_hash', 
        aggfunc='count', 
        fill_value=0
    )
    print(team_class_pivot)
    
    # --- File Size Analysis ---
    print(f"\n📊 File Size Analysis:")
    print(f"Original Dataset:")
    print(f"  • Mean file size: {df_clean_summary['old_size_mb'].mean():.2f} MB")
    print(f"  • Total original size: {df_clean_summary['old_size_mb'].sum():.2f} MB")
    print(f"Resized Dataset:")
    print(f"  • Mean file size: {df_clean_summary['new_image_size'].mean():.2f} MB")
    print(f"  • Total resized size: {df_clean_summary['new_image_size'].sum():.2f} MB")
    compression_ratio = (1 - df_clean_summary['new_image_size'].sum() / df_clean_summary['old_size_mb'].sum()) * 100
    print(f"  • Compression ratio: {compression_ratio:.1f}%")
    
    # --- Resolution Analysis ---
    print(f"\n📊 Original Resolution Analysis:")
    print(f"  • Mean width: {df_clean_summary['old_width'].mean():.0f}px")
    print(f"  • Mean height: {df_clean_summary['old_height'].mean():.0f}px")
    print(f"  • Min width: {df_clean_summary['old_width'].min()}px")
    print(f"  • Max width: {df_clean_summary['old_width'].max()}px")
    print(f"  • Min height: {df_clean_summary['old_height'].min()}px")
    print(f"  • Max height: {df_clean_summary['old_height'].max()}px")
    
    # --- Extra Subfolders Analysis ---
    if df_summary['Extra_subfolder'].any():
        print(f"\n📊 Extra Subfolders Found:")
        subfolder_counts = df_summary[df_summary['Extra_subfolder'] != '']['Extra_subfolder'].value_counts()
        for subfolder, count in subfolder_counts.items():
            print(f"  • '{subfolder}': {count} images")
    
    # --- File Format Analysis ---
    print(f"\n📊 Original File Format Distribution:")
    format_counts = df_clean_summary['old_extension'].value_counts()
    for ext, count in format_counts.items():
        print(f"  • {ext}: {count} images")
    
    # --- Validation Checks ---
    print(f"\n✅ Validation Checks:")
    
    # Check for missing files
    missing_files = 0
    for _, row in df_clean_summary.iterrows():
        if not os.path.exists(row['new_image_path']):
            missing_files += 1
    
    if missing_files == 0:
        print(f"  ✅ All {len(df_clean_summary)} processed images exist in output directory")
    else:
        print(f"  ❌ {missing_files} processed images are missing from output directory")
    
    # Check directory structure
    expected_teams = df_clean_summary['team'].unique()
    expected_classes = df_clean_summary['class'].unique()
    
    print(f"  ✅ Directory structure validation:")
    for team in expected_teams:
        team_dir = os.path.join(OUTPUT_ROOT_DIR, team)
        if os.path.exists(team_dir):
            team_classes = df_clean_summary[df_clean_summary['team'] == team]['class'].unique()
            for cls in team_classes:
                class_dir = os.path.join(team_dir, cls)
                if os.path.exists(class_dir):
                    file_count = len([f for f in os.listdir(class_dir) if f.endswith(OUTPUT_EXTENSION)])
                    expected_count = len(df_clean_summary[(df_clean_summary['team'] == team) & (df_clean_summary['class'] == cls)])
                    if file_count == expected_count:
                        print(f"    ✅ {team}/{cls}: {file_count} files")
                    else:
                        print(f"    ❌ {team}/{cls}: {file_count} files (expected {expected_count})")
                else:
                    print(f"    ❌ Missing directory: {team}/{cls}")
        else:
            print(f"    ❌ Missing team directory: {team}")
    
    # --- Export Summary Statistics ---
    stats_summary = {
        'total_images_processed': len(df_clean_summary),
        'total_images_corrupted': len(df_summary) - len(df_clean_summary),
        'total_teams': df_clean_summary['team'].nunique(),
        'total_classes': df_clean_summary['class'].nunique(),
        'original_total_size_mb': df_clean_summary['old_size_mb'].sum(),
        'resized_total_size_mb': df_clean_summary['new_image_size'].sum(),
        'compression_ratio_percent': compression_ratio,
        'target_resolution': f"{TARGET_SIZE}x{TARGET_SIZE}",
        'output_format': OUTPUT_FORMAT
    }
    
    stats_df = pd.DataFrame([stats_summary])
    stats_path = os.path.join(OUTPUT_ROOT_DIR, "dataset_statistics.csv")
    stats_df.to_csv(stats_path, index=False)
    print(f"\n📊 Dataset statistics saved to '{stats_path}'")
    
else:
    print("❌ No summary data available for analysis")

print(f"\n🔍 Analysis Complete!")

🔍 Dataset Analysis and Validation

📊 Team Distribution:
  • AI-4o: 7344 images
  • PLT: 6556 images
  • SMART AGRICULTURES: 6321 images
  • condimenteum: 5504 images
  • CACTUS: 3801 images
  • GreenAI: 3785 images
  • AiGro: 3633 images
  • RUSTICUS: 3446 images
  • Scorpions: 2795 images
  • CHAJARA: 2540 images
  • the Neural Ninjas: 1642 images

📊 Class Distribution:
  • chene: 8471 images
  • faux_poivrier: 8064 images
  • tipu: 8034 images
  • frene: 7953 images
  • pistachier: 7518 images
  • caroubier: 7327 images

📊 Team-Class Distribution:
class               caroubier  chene  faux_poivrier  frene  pistachier  tipu
team                                                                        
AI-4o                     960   1625           1221    934        1111  1493
AiGro                     686    604            595    572         541   635
CACTUS                    400    597            694    803         755   552
CHAJARA                   610    549            428    308 